# 02. Inner-product geometry and numerical stability

![Vectors and stable cosine similarity](../images/02_inner_product_geometry.svg)

**Learning goals:** connect dot products to angles, compute norms and cosine similarity, pool token features, recognize floating-point failure modes, and implement explicit zero-norm and tolerance policies.

In [ ]:
import random
import numpy as np
import torch
import torch.nn.functional as F

SEED = 11
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print(f'numpy={np.__version__}, torch={torch.__version__}, device=cpu')

## 1. Dot products and norms

The dot product $x^T y=\sum_i x_i y_i$ measures signed alignment. The Euclidean norm is $\|x\|_2=\sqrt{x^T x}$. Their geometric identity is $x^T y=\|x\|\|y\|\cos\theta$.

In [ ]:
x = np.array([3.0, 4.0])
y = np.array([-4.0, 3.0])
dot = x @ y
norm_x, norm_y = np.linalg.norm(x), np.linalg.norm(y)
cosine = dot / (norm_x * norm_y)
assert dot == 0 and norm_x == norm_y == 5
assert np.isclose(cosine, 0.0)
print(f'dot={dot:.1f}, norms=({norm_x:.1f}, {norm_y:.1f}), cosine={cosine:.1f}')

## 2. Cosine similarity removes magnitude

Normalize each nonzero vector to unit length, then take a dot product. For many vectors, `X_normalized @ X_normalized.T` computes every pair using optimized matrix multiplication rather than Python loops.

In [ ]:
X = np.array([[1.0, 0.0], [1.0, 1.0], [-2.0, 0.0]])
Xn = X / np.linalg.norm(X, axis=-1, keepdims=True)
S = Xn @ Xn.T
np.testing.assert_allclose(np.diag(S), 1.0, atol=1e-12)
np.testing.assert_allclose(S, S.T, atol=1e-12)
assert np.isclose(S[0, 2], -1.0)
print(np.round(S, 3))

## 3. Mean and masked pooling

For `(B,N,D)` token features, `mean(dim=1)` removes only the token axis. Padded tokens need a mask so they do not dilute the average. Clamping the count prevents division by zero, but an all-masked row should still be reported explicitly.

In [ ]:
tokens = torch.arange(2*4*3, dtype=torch.float32).reshape(2, 4, 3)
mask = torch.tensor([[True, True, False, False], [True, True, True, False]])
weights = mask.unsqueeze(-1).to(tokens.dtype)
counts = weights.sum(dim=1).clamp_min(1.0)
masked_mean = (tokens * weights).sum(dim=1) / counts
assert masked_mean.shape == (2, 3)
torch.testing.assert_close(masked_mean[0], tokens[0, :2].mean(dim=0))
assert mask.any(dim=1).all()
print('masked means:\n', masked_mean)

## 4. Floating-point range and precision

`float32` has limited exponent range and about seven decimal digits of precision. Squaring a large finite value can overflow before the final square root. Long reductions also accumulate rounding error. Use library norms and promote sensitive reductions when the added cost is justified.

In [ ]:
large32 = np.array([1e20, 1e20], dtype=np.float32)
with np.errstate(over='ignore'):
    naive = np.sqrt(np.sum(large32 * large32))
promoted = np.sqrt(np.sum(large32.astype(np.float64) ** 2))
assert np.isinf(naive) and np.isfinite(promoted)

# Adding a tiny value to a much larger float32 value can round away.
rounded = np.float32(1e8) + np.float32(1.0)
assert rounded == np.float32(1e8)
print(f'naive={naive}, promoted={promoted:.3e}, rounded={rounded:.1f}')

## 5. A zero-vector policy

Cosine similarity is undefined for a zero vector. Clamping the denominator is a computational convention: it maps zero to zero and gives similarity zero with every vector. `F.normalize` implements stable feature-axis normalization without constructing a large broadcasted pair tensor.

In [ ]:
features = torch.tensor([[3.0, 4.0], [0.0, 0.0], [1e-12, 0.0]])
normalized = F.normalize(features, dim=-1, eps=1e-8)
assert torch.isfinite(normalized).all()
torch.testing.assert_close(normalized[0], torch.tensor([0.6, 0.8]))
torch.testing.assert_close(normalized[1], torch.zeros(2))
pairwise = normalized @ normalized.T
print('stable normalized vectors:\n', normalized)
print('pairwise similarities:\n', pairwise)

## 6. Test with relative and absolute tolerances

Approximate equality checks $|a-b| \le atol + rtol|b|$. Absolute tolerance matters near zero; relative tolerance scales with magnitude. Choose both from the dtype, reduction length, and consequence of the comparison.

In [ ]:
a = np.float64(0.1) + np.float64(0.2)
b = np.float64(0.3)
assert a != b
assert np.isclose(a, b, rtol=1e-12, atol=1e-15)

random_features = torch.randn(64, 32)
unit = F.normalize(random_features, dim=-1)
torch.testing.assert_close(torch.linalg.vector_norm(unit, dim=-1), torch.ones(64))
assert torch.isfinite(unit).all()
print('0.1 + 0.2 error:', format(a - b, '.3e'))

## Exercises and final takeaways

**Exercises:** (1) Compute the cosine similarity of `(1,2,2)` and `(2,0,1)` by hand, then verify it. (2) Add an all-masked sequence and return both the safe mean and an `is_valid` flag. (3) Compare pairwise cosine results in `float32` and `float64` for nearly parallel vectors.

**Takeaways:** dot products mix length and alignment; cosine similarity isolates direction; pooling must reduce the intended axis; stable software needs explicit dtype, epsilon, finite-value, and tolerance policies.

## Continue learning

[Previous notebook: 01](01_spatiotemporal_tensor_geometry.ipynb) | [Lecture](../lectures/02_inner_product_geometry.md) | [Curriculum](../README.md) | [Next notebook: 03](03_hierarchical_observations.ipynb)